In [9]:
import boto3
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, timezone
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler

# -----------------------------
# CONFIG
# -----------------------------
REGION = "us-east-2"  # change if needed
INSTANCE_ID = "i-0372640781068e64b"  # 🔴 REPLACE THIS

# -----------------------------
# AWS CLIENT
# -----------------------------
cloudwatch = boto3.client("cloudwatch", region_name=REGION)

# -----------------------------
# Fetch metric safely
# -----------------------------
def get_metric(metric_name):
    response = cloudwatch.get_metric_statistics(
        Namespace="AWS/EC2",
        MetricName=metric_name,
        Dimensions=[{"Name": "InstanceId", "Value": INSTANCE_ID}],
        StartTime=datetime.now(timezone.utc) - timedelta(hours=24),
        EndTime=datetime.now(timezone.utc),
        Period=300,
        Statistics=["Average"]
    )
    return response.get("Datapoints", [])

# -----------------------------
# Convert to DataFrame safely
# -----------------------------
def to_df(data, col_name):
    if not data:
        print(f"No data for {col_name}")
        return pd.DataFrame(columns=["Timestamp", col_name])

    df = pd.DataFrame(data)

    if "Timestamp" not in df.columns or "Average" not in df.columns:
        print(f"Unexpected format for {col_name}")
        print(df.head())
        return pd.DataFrame(columns=["Timestamp", col_name])

    df = df.sort_values("Timestamp")
    df = df[["Timestamp", "Average"]]
    df.rename(columns={"Average": col_name}, inplace=True)

    return df

# -----------------------------
# FETCH DATA
# -----------------------------
cpu_data = get_metric("CPUUtilization")
net_in_data = get_metric("NetworkIn")
net_out_data = get_metric("NetworkOut")

print("CPU datapoints:", len(cpu_data))
print("NetworkIn datapoints:", len(net_in_data))
print("NetworkOut datapoints:", len(net_out_data))

# -----------------------------
# BUILD DATAFRAMES
# -----------------------------
df_cpu = to_df(cpu_data, "cpu_usage_pct")
df_in = to_df(net_in_data, "network_in_mb")
df_out = to_df(net_out_data, "network_out_mb")

# -----------------------------
# MERGE SAFELY
# -----------------------------
df = pd.merge(df_cpu, df_in, on="Timestamp", how="outer")
df = pd.merge(df, df_out, on="Timestamp", how="outer")

# Fill missing values
df = df.sort_values("Timestamp").reset_index(drop=True)
df = df.ffill().bfill()

# -----------------------------
# CONVERT BYTES → MB
# -----------------------------
if "network_in_mb" in df:
    df["network_in_mb"] /= (1024 * 1024)

if "network_out_mb" in df:
    df["network_out_mb"] /= (1024 * 1024)

# -----------------------------
# SIMULATE MISSING FEATURES
# -----------------------------


# -----------------------------
# COST MODEL (simple)
# -----------------------------


# -----------------------------
# HANDLE EDGE CASE: too little data
# -----------------------------
if len(df) < 10:
    raise ValueError("Not enough data points. Wait a few minutes for CloudWatch metrics.")

# -----------------------------
# Z-SCORE ANOMALY DETECTION
# -----------------------------
df["z_score"] = np.abs(stats.zscore(df["cpu_usage_pct"]))
df["is_anomaly"] = (df["z_score"] > 0.1).astype(int)
print(df["is_anomaly"].value_counts())

# -----------------------------
# PREPARE ML DATA
# -----------------------------
X = df[[
    "cpu_usage_pct",
    "network_in_mb",
    "network_out_mb"
    
]].values

y = df["is_anomaly"].values

# -----------------------------
# TRAIN-TEST SPLIT
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.5, random_state=42, stratify=y
) 
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"\nTraining samples : {len(X_train)}")
print(f"Testing samples  : {len(X_test)}")
print(f"Anomalies in training set: {y_train.sum()}")
print(f"Anomalies in test set    : {y_test.sum()}")
print("Anomaly distribution:\n", df["is_anomaly"].value_counts())
print("Newest timestamp:", df["Timestamp"].max())

# -----------------------------
# TRAIN MODEL
# -----------------------------
model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train, y_train)


# -----------------------------
# PREDICT
# -----------------------------
y_pred = model.predict(X_test)

# -----------------------------
# EVALUATE
# -----------------------------
accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print(f"\nModel Accuracy: {accuracy * 100:.2f}%")
print("\nConfusion Matrix:")
print(cm)

# -----------------------------
# FEATURE IMPORTANCE
# -----------------------------
feature_names = ["cpu_usage_pct", "network_in_mb", "network_out_mb"]

print("\nFeature Importance:")
for name, coef in zip(feature_names, model.coef_[0]):
    print(f"{name:25s}: {coef:.6f}")

# -----------------------------
# FINAL CHECK
# -----------------------------
print("\nSample Data:")
print(df.tail())

CPU datapoints: 28
NetworkIn datapoints: 28
NetworkOut datapoints: 28
is_anomaly
1    26
0     2
Name: count, dtype: int64

Training samples : 14
Testing samples  : 14
Anomalies in training set: 13
Anomalies in test set    : 13
Anomaly distribution:
 is_anomaly
1    26
0     2
Name: count, dtype: int64
Newest timestamp: 2026-03-27 23:46:00+00:00

Model Accuracy: 92.86%

Confusion Matrix:
[[ 1  0]
 [ 1 12]]

Feature Importance:
cpu_usage_pct            : 0.012359
network_in_mb            : -1.470288
network_out_mb           : 0.065260

Sample Data:
                  Timestamp  cpu_usage_pct  network_in_mb  network_out_mb  \
0 2026-03-27 21:31:00+00:00       3.404246      18.801038        0.032615   
1 2026-03-27 21:36:00+00:00       0.163324       0.000762        0.000820   
2 2026-03-27 21:41:00+00:00       0.136613       0.000286        0.000341   
3 2026-03-27 21:46:00+00:00       2.770015       0.267832        0.006281   
4 2026-03-27 21:51:00+00:00       7.379452      38.604953    

/home/ec2-user/.local/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)
